# Notebook 03 — Prompt Engineering

> **阶段**：Stage 2 Experiment · **预计时间**：30–60 分钟（CPU 用小样本） · **平台**：Kaggle Notebook

Prompt 本身就是实验变量。本 Notebook 用固定页面集合对比 v0–v3 的行为差异。


# Learning Objectives

- 理解为什么 Prompt 必须与数据、采样一样被固定和记录；
- 在固定 subset 上运行多个 prompt，比较输出与延迟；
- 建立 Prompt Benchmark Table（文本/表格/公式/阅读顺序的粗观察 + latency）。


# Why This Matters

模型能力上限由权重决定，但实际表现受 Prompt 影响。不记录 prompt_id 的实验无法复现，也无法归因——这是之后所有消融实验的纪律基础。


# Concepts

- v0：官方默认 full conversion；v1：强调 OCR；v2：强调阅读顺序/表格/公式/版面；v3：结构化解析指令；
- 每次推理记录 prompt_id / model revision / image_id / generation_config / latency；
- CPU 教学环境：本 Notebook 默认只用 3 页 × 2 个 prompt（约 1–1.5 h），GPU 上可扩到全部。


## Step 1 — 固定 subset（与后续实验同 seed 同页面）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.config import load_config

cfg = load_config()
data_root = data.find_dataset_root()
annotations = data.load_annotations(data_root)
subset = data.select_pages(annotations, n=3, seed=42)
print('固定页面:', [data.sample_id(p) for p in subset])


## Step 2 — 依次运行各 Prompt（缓存自动复用）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.config import project_root
from src.inference import run_baseline
from src.model import SmolDoclingAdapter

PROMPTS_TO_RUN = ['v0', 'v1']  # TODO: GPU 环境扩展到 v0-v3
adapter = SmolDoclingAdapter().load()

manifests = {}
for pid in PROMPTS_TO_RUN:
    out = project_root() / 'results' / ('prompt_' + pid)
    m = run_baseline(
        subset, data_root, adapter, output_dir=out,
        mode='fast', config=cfg, prompt_id=pid, skip_existing=True, n_pages=len(subset),
    )
    manifests[pid] = m
    print(pid, '->', m)


## Step 3 — Prompt Benchmark Table


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import json
import pandas as pd
from src import data, evaluation

rows = []
for pid, m in manifests.items():
    summary = data.read_json(m.parent / 'summary.json')
    preds = [json.loads(line) for line in m.read_text(encoding='utf-8').splitlines() if line.strip()]
    sanity = evaluation.sanity_check(m.parent / 'predictions', annotations, data_root)
    overall = sum(r['sanity_ned'] for r in sanity) / len(sanity) if sanity else None
    rows.append({
        'prompt': pid,
        'pages': summary['completed'],
        'mean_latency_sec': summary['mean_latency_sec'],
        'mean_doctags_chars': round(sum(len(p['doctags']) for p in preds) / len(preds), 1) if preds else None,
        'sanity_ned_mean': round(overall, 4) if overall is not None else None,
        'metric_kind': 'non_official_smoke',
    })
bench = pd.DataFrame(rows)
display(bench)
data.write_json(rows, project_root() / 'results' / 'prompt_benchmark.json')


# What You Should Observe

- 不同 Prompt 下输出长度与延迟可能明显不同（长输出 = 更多 token = 更慢）；
- sanity_ned 只是文本层面的粗信号；表格/公式/阅读顺序的差异要看 doctags 本身，正式结论必须等 Notebook 07 的官方指标；
- 同一页面不同 prompt 的结果都带 prompt_id，可逐条对比。


# Research Checkpoint

> **Prompt 如何成为实验变量？** 用你今天的输出举例：哪个 Prompt 在哪类内容上行为不同？这如何影响你把 Prompt 写进实验设计（唯一变量原则）？

**TODO：** 答案写入 `results/nb03/research_checkpoint.md`。


# Exercises

1. **TODO：** 找出两个 prompt 在**同一页面**上 doctags 差异最大的地方（例如表格/公式标记），说明差异来源；
2. **TODO：** 给 v3 写一个你自己的变体 v4（放在 src/prompts.py），跑同一 subset 并加进对比表；
3. **TODO：** 解释为什么「换了 prompt 之后分数变好」不能直接归因于「模型变强」。


# Takeaways

- Prompt 与数据、采样同属实验变量，必须记录 prompt_id；
- 输出长度影响延迟：Prompt 评测表同时看内容与成本；
- 正式指标在 Notebook 07，这里只做受控观察。

**下一步**：[Notebook 05](05_SFT_Fundamentals.ipynb) — Fine-tuning 改变了什么。
